<a href="https://colab.research.google.com/github/IvanRosasCCT/Gibtest/blob/main/01_IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import time
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Library loads")
print("TensorFlow version:", tf.__version__)
print(f'GPU avalible: {tf.config.list_physical_devices("GPU")}')


Library loads
TensorFlow version: 2.19.0
GPU avalible: []


In [2]:
# conect Google Drive

from google.colab import drive
drive.mount('/content/drive')
print("Drive connected")

Mounted at /content/drive
Drive connected


In [3]:
#load CICIDS2017 from Driver

FILE_PATH  = '/content/drive/MyDrive/Thesis CICIDS2017/wednesday.csv'

df = pd.read_csv(FILE_PATH, low_memory= False)
df.head()


,Src IP dec,Src Port,Dst IP dec,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,ICMP Code,ICMP Type,Total TCP Flow Time,Label,Attempted Category
0,3232238089,13793,885387958,443,6,43:34.8,187889,1,1,0,...,0,0.0,0.0,0,0,-1,-1,187889,BENIGN,-1
1,3232238089,13793,885387958,443,6,43:35.5,187758,1,1,0,...,0,0.0,0.0,0,0,-1,-1,187758,BENIGN,-1
2,3232238089,13794,910236785,443,6,43:36.4,189882,1,1,0,...,0,0.0,0.0,0,0,-1,-1,189882,BENIGN,-1
3,3232238089,13794,910236785,443,6,43:37.1,190117,1,1,0,...,0,0.0,0.0,0,0,-1,-1,190117,BENIGN,-1
4,3232238089,13796,885387958,443,6,43:38.0,188603,1,1,0,...,0,0.0,0.0,0,0,-1,-1,188603,BENIGN,-1


In [13]:
print(f'Rows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')


Rows: 496640
Columns: 89


In [14]:
# initial inspection

print('Label Distribution')
print (df['Label'].value_counts())

print('Null Values')
null_count = df.isnull().sum()
null_cols = null_count[null_count > 0]
if len(null_cols) > 0:
  print(null_cols)
else:
  print('No null values')

print('Timestamp column:')
if 'Timestamp' in df.columns:
    print(f'Data Type: {df["Timestamp"].dtype}')
    print(f'Unique Values: {df["Timestamp"].nunique()}')
    print(f'First Value: {df["Timestamp"].iloc[0]}')
    print(f'Last Value: {df["Timestamp"].iloc[-1]}')
else:
    print('No timestamp column found')



Label Distribution
Label
BENIGN                          319119
DoS Hulk                        158468
DoS GoldenEye                     7567
DoS Slowloris                     3859
DoS Slowhttptest - Attempted      3368
DoS Slowloris - Attempted         1847
DoS Slowhttptest                  1740
DoS Hulk - Attempted               581
DoS GoldenEye - Attempted           80
Heartbleed                          11
Name: count, dtype: int64
Null Values
No null values
Timestamp column:
Data Type: object
Unique Values: 30857
First Value: 43:34.8
Last Value: 59:58.7


In [15]:
# Binary Labeling
# 0 = Bening
# 1 = Attack

#copy dataset
df_copy = df.copy()

#original labels
print(df_copy['Label'].value_counts())

#Everything that is NOT Bening is considered Attack

def binary_label(labe):
  if labe == 'BENIGN':
    return 0  #Normal
  else:
    return 1  #Attack

df_copy['Attack'] = df_copy['Label'].apply(binary_label)

print('\n New distribution')
print(df_copy['Attack'].value_counts())

print(f"\nAttack Percentage:{df_copy['Attack'].mean()*100:.2f}%")




Label
BENIGN                          319119
DoS Hulk                        158468
DoS GoldenEye                     7567
DoS Slowloris                     3859
DoS Slowhttptest - Attempted      3368
DoS Slowloris - Attempted         1847
DoS Slowhttptest                  1740
DoS Hulk - Attempted               581
DoS GoldenEye - Attempted           80
Heartbleed                          11
Name: count, dtype: int64

 New distribution
Attack
0    319119
1    177521
Name: count, dtype: int64

Attack Percentage:35.74%


In [16]:
# Temporal preparation
print("Temporal preparation")
print(f'Data Type: {df_copy["Timestamp"].dtype}')
print(f'Unique Values: {df_copy["Timestamp"].nunique()}')
print("First 5 values:")
print(df_copy["Timestamp"].head().tolist())

df_copy = df_copy.reset_index(drop=True)
df_copy["TimeOrder"] = df_copy.index

print("\nDataSet ordered chronologically")
print(f"Total Flows: {len(df_copy):,} ")
print(f"First index: {df_copy['TimeOrder'].iloc[0]}")
print(f"Last index: {df_copy['TimeOrder'].iloc[-1]}")

print("\nTemporal Verification")
print(f"Initial Timestamo: {df_copy['Timestamp'].iloc[0]}")
print(f"Final Timestamp: {df_copy['Timestamp'].iloc[-1]}")







Temporal preparation
Data Type: object
Unique Values: 30857
First 5 values:
['43:34.8', '43:35.5', '43:36.4', '43:37.1', '43:38.0']

DataSet ordered chronologically
Total Flows: 496,640 
First index: 0
Last index: 496639

Temporal Verification
Initial Timestamo: 43:34.8
Final Timestamp: 59:58.7


In [17]:
print("Temporal Windows")
#Windows parameters
WINDOW_SIZE = 1000
STEP_SIZE = 500

windows = []
total_rows = len(df_copy)
n_windows = 0

print(f"Total rows: {total_rows:,}")
print(f"Window size: {WINDOW_SIZE}")
print(f"Step size: {STEP_SIZE}")

#  windows sliding

for start in range(0, total_rows - WINDOW_SIZE + 1, STEP_SIZE):
    end = start + WINDOW_SIZE
    window_df = df_copy.iloc[start:end]

    total_fwd_pkts = window_df["Total Fwd Packet"].sum()
    if total_fwd_pkts == 0:
      total_fwd_pkts = 1

    #Extract 12 feactures per windows
    features = {

        #
        # F1 packet Rate
        "Packet_Rate": window_df["Total Fwd Packet"].sum() / WINDOW_SIZE,
        # F2 Byte Rate
        "Bytes_Rate": window_df["Total Length of Fwd Packet"].sum() / WINDOW_SIZE,
        #F3 SYN Ratio
        "SYN_Ratio": window_df["SYN Flag Count"].sum() / total_fwd_pkts,
        #F4 ACK Ratio
        "ACK_Ratio": window_df["ACK Flag Count"].sum() / total_fwd_pkts,
        #F5 Average Flow Duration
        "Avg_Flow_Duration": window_df["Flow Duration"].mean(),
        #F6 Unique Source IPs
        "Unique_Source_IPs": window_df["Src IP dec"].nunique(),
        #F7 Destination Port Entropy
        "Dst_Port_Entropy": window_df["Dst Port"].nunique() / WINDOW_SIZE,
        #F8 - F11 Packet Statistics
        "Pkt_Mean": window_df["Total Fwd Packet"].mean(),
        "Pkt_Std": window_df["Total Fwd Packet"].std(),
        "Pkt_Min": window_df["Total Fwd Packet"].min(),
        "Pkt_Max": window_df["Total Fwd Packet"].max(),
        #F12 Flow cpunt
        "Flow_Count": window_df.shape[0],
        #Binary label: 1 if at least one attack exists in window
        "Label": 1 if window_df["Attack"].sum() > 0 else 0
    }
    windows.append(features)
    n_windows += 1

#Windows DataFrame
df_windows = pd.DataFrame(windows)
print(f"Total windows: {n_windows:,}")

print("\nLabel Distribution")
print(df_windows["Label"].value_counts())
print(f"\nPercentage of attacks per windows: {df_windows['Label'].mean()*100:.2f}%")

print("Ratios Verification:")
print(f"SYN_Ratio min/max {df_windows['SYN_Ratio'].min():.4f} / {df_windows['SYN_Ratio'].max():.4f}")
print(f"ACK Ratio min/max {df_windows['ACK_Ratio'].min():.4f} / {df_windows['ACK_Ratio'].max():.4f}")


print("First 3 Windows:")
print(df_windows.head(3))
print("\nLast 3 Windows:")
print(df_windows.tail(3))

Temporal Windows
Total rows: 496,640
Window size: 1000
Step size: 500
Total windows: 992

Label Distribution
Label
1    839
0    153
Name: count, dtype: int64

Percentage of attacks per windows: 84.58%
Ratios Verification:
SYN_Ratio min/max 0.0003 / 0.8211
ACK Ratio min/max 0.0250 / 2.4334
First 3 Windows:
   Packet_Rate  Bytes_Rate  SYN_Ratio  ACK_Ratio  Avg_Flow_Duration  \
0      854.581    2458.998   0.000618   2.278944       4.747939e+07   
1      907.045    2051.439   0.000615   2.318587       4.855092e+07   
2     1008.719     820.280   0.000341   2.341740       3.740846e+07   

   Unique_Source_IPs  Dst_Port_Entropy  Pkt_Mean       Pkt_Std  Pkt_Min  \
0                 14             0.015   854.581  11905.614872        1   
1                 16             0.013   907.045  12739.625438        1   
2                 16             0.013  1008.719  14101.763519        1   

   Pkt_Max  Flow_Count  Label  
0   203943        1000      0  
1   202455        1000      0  
2   202455

In [9]:
#Check Available Columns

print("Flag-Related columns")
flag_cols = [col for col in df_copy.columns if "flag" in col.lower() or "syn" in col.lower() or "ack" in col.lower()]
print(flag_cols)

print("\nFist values of those columns;")
if flag_cols:
    print(df_copy[flag_cols].head())
else:
    print("No flag-related columns found in the DataFrame.")

print("All collumns (first 20):")
print(df_copy.columns[:20].tolist())

Flag-Related columns
['Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Packets/s', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Average Packet Size', 'Fwd Packet/Bulk Avg', 'Bwd Packet/Bulk Avg', 'Subflow Fwd Packets', 'Subflow Bwd Packets', 'Attack']

Fist values of those columns;
   Total Fwd Packet  Total Bwd packets  Total Length of Fwd Packet  \
0                 1                  1  

In [10]:
from sklearn.utils import class_weight

#from sklearn.preprocessing import RobustScaler
from sklearn.utils.class_weight import compute_class_weight

print("Scaling and Train / Test Split")

#Separate features (x) and label (y)

features_cols = [c for c in df_windows.columns if c != 'Label']
X = df_windows[features_cols].values
y = df_windows['Label'].values


print(f"Features: {len(features_cols)}")
print(f"Names:{features_cols}")

# 70% Train, 30% Test

split_idx = int(len(df_windows) * 0.7)
X_train_raw = X[:split_idx]
y_train = y[:split_idx]
X_test_raw = X[split_idx:]
y_test = y[split_idx:]

print("\nSplit:")

print(f"Train: {len(X_train_raw)} windows ({len(X_train_raw)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test_raw)} windows ({len(X_test_raw)/len(X)*100:.1f}%)")

print(f"\nTrain Labels: {np.bincount(y_train)}")
print(f"Test Labels: {np.bincount(y_test)}")

#RobustScaler

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("\nScaling completed")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"\n Class Weights")
print(f"Class 0 (Benign): {class_weight_dict[0]:.4f}")
print(f"Class 1 (Attack): {class_weight_dict[1]:.4f}")




Scaling and Train / Test Split
Features: 12
Names:['Packet_Rate', 'Bytes_Rate', 'SYN_Ratio', 'ACK_Ratio', 'Avg_Flow_Duration', 'Unique_Source_IPs', 'Dst_Port_Entropy', 'Pkt_Mean', 'Pkt_Std', 'Pkt_Min', 'Pkt_Max', 'Flow_Count']

Split:
Train: 694 windows (70.0%)
Test: 298 windows (30.0%)

Train Labels: [153 541]
Test Labels: [  0 298]

Scaling completed
X_train Shape: (694, 12)
X_test Shape: (298, 12)

 Class Weights
Class 0 (Benign): 2.2680
Class 1 (Attack): 0.6414


In [11]:
from enum import unique
# Find optimal Split

labels_array = df_windows["Label"].values
total = len(labels_array)

for pct in [0.40,0.45,0.50,0.55,0.60,0.65]:
  idx = int(total * pct)
  test_label = labels_array[idx:]

  n_benign = (test_label == 0).sum()
  n_attack = (test_label == 1).sum()

  print(f"\nSplit: {pct*100:.0f}% Train / {(1-pct)*100:.0f}% Test:")
  print(f"Test size: {len(test_label)} windows")
  print(f"Benign (0): {n_benign}")
  print(f"Attack (1): {n_attack}")

  if n_benign > 0 and n_attack > 0:
    unique, counts = np.unique(test_label, return_counts=True)
    print(f"Both Classes Presents")
    best_split = pct
    best_idx = idx
  else:
    print(f"Only Class: {unique[0]} ({counts[0]})")

print(f"\nBest split: {best_split*100:.0f}%")




Split: 40% Train / 60% Test:
Test size: 596 windows
Benign (0): 125
Attack (1): 471
Both Classes Presents

Split: 45% Train / 55% Test:
Test size: 546 windows
Benign (0): 75
Attack (1): 471
Both Classes Presents

Split: 50% Train / 50% Test:
Test size: 496 windows
Benign (0): 36
Attack (1): 460
Both Classes Presents

Split: 55% Train / 45% Test:
Test size: 447 windows
Benign (0): 0
Attack (1): 447
Only Class: 0 (36)

Split: 60% Train / 40% Test:
Test size: 397 windows
Benign (0): 0
Attack (1): 397
Only Class: 0 (36)

Split: 65% Train / 35% Test:
Test size: 348 windows
Benign (0): 0
Attack (1): 348
Only Class: 0 (36)

Best split: 50%


In [18]:
print("Repeat Scaling and Train / Test Split whit 50/50")

#Separate features (x) and label (y)

features_cols = [c for c in df_windows.columns if c != 'Label']
X = df_windows[features_cols].values
y = df_windows['Label'].values


print(f"Features: {len(features_cols)}")
print(f"Names:{features_cols}")

# 50% Train, 50% Test

split_idx = int(len(df_windows) * 0.5)
X_train_raw = X[:split_idx]
y_train = y[:split_idx]
X_test_raw = X[split_idx:]
y_test = y[split_idx:]

print("\nSplit:")

print(f"Train: {len(X_train_raw)} windows (first 50%, morning/noon)")
print(f"Test: {len(X_test_raw)} windows (last 50%, afternoon)")
print(f"\nTrain Benign: {(y_train==0).sum()}, Attack: {(y_train==1).sum()}")
print(f"\nTest Benign: {(y_test==0).sum()}, Attack: {(y_test==1).sum()}")

#RobustScaler

scaler = RobustScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print("\nScaling completed")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"\n Class Weights")
print(f"Class 0 (Benign): {class_weight_dict[0]:.4f}")
print(f"Class 1 (Attack): {class_weight_dict[1]:.4f}")

Repeat Scaling and Train / Test Split whit 50/50
Features: 12
Names:['Packet_Rate', 'Bytes_Rate', 'SYN_Ratio', 'ACK_Ratio', 'Avg_Flow_Duration', 'Unique_Source_IPs', 'Dst_Port_Entropy', 'Pkt_Mean', 'Pkt_Std', 'Pkt_Min', 'Pkt_Max', 'Flow_Count']

Split:
Train: 496 windows (first 50%, morning/noon)
Test: 496 windows (last 50%, afternoon)

Train Benign: 117, Attack: 379

Test Benign: 36, Attack: 460

Scaling completed
X_train Shape: (496, 12)
X_test Shape: (496, 12)

 Class Weights
Class 0 (Benign): 2.1197
Class 1 (Attack): 0.6544


In [20]:
# Create LSTM Sequence

# Ensure y is 1D (Flat)
y_train_flat = np.array(y_train).flatten()
y_test_flat = np.array(y_test).flatten()

print(f"y_train_flat Shape: {y_train_flat.shape}")
print(f"y_test_flat Shape: {y_test_flat.shape}")

SEQ_LENGTH = 12

def create_sequences(X,y,seq_length):
  X_seq, y_seq = [],[]
  # X: Scaled data
  # y: Labels

  for i in range(len(X) - seq_length + 1):
      X_seq.append(X[i:i+seq_length])

      last_label = y[i+seq_length-1]
      y_seq.append(int(last_label))

  return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train, y_train, SEQ_LENGTH)
X_test_seq, y_test_seq = create_sequences(X_test, y_test, SEQ_LENGTH)

print(f"Sequence length: {SEQ_LENGTH} windows")
print(f"\n Train Sequences:")
print(f"X_train_seq Shape: {X_train_seq.shape}")
print(f"y_train_seq Shape: {y_train_seq.shape}")
print(f"Distribution: Benign = {(y_train_seq ==0).sum()}, Attack = {(y_train_seq ==1).sum()}")

print(f"\n Test Sequences:")
print(f"X_test_seq Shape: {X_test_seq.shape}")
print(f"y_test_seq Shape: {y_test_seq.shape}")
print(f"Distribution: Benign = {(y_test_seq ==0).sum()}, Attack = {(y_test_seq ==1).sum()}")



y_train_flat Shape: (496,)
y_test_flat Shape: (496,)
Sequence length: 12 windows

 Train Sequences:
X_train_seq Shape: (485, 12, 12)
y_train_seq Shape: (485,)
Distribution: Benign = 106, Attack = 379

 Test Sequences:
X_test_seq Shape: (485, 12, 12)
y_test_seq Shape: (485,)
Distribution: Benign = 32, Attack = 453


In [25]:
# LSTM Baseline Model

from tensorflow.keras.metrics import Precision,Recall

print("\nBuilding LSTM Baseline")

# Clean TensorFlow session
tf.keras.backend.clear_session()

# Model
model_lstm = Sequential([
    LSTM(64, input_shape=(SEQ_LENGTH, X_train_seq.shape[2]),
         activation= 'tanh', recurrent_activation='sigmoid'),
    # Dropout for regularization (prevents overfitting)
    Dropout(0.2),
    # Output layer: 1 neuron with sigmoid (attack propability)
    Dense(1, activation='sigmoid')

])

# compile model
model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),  # Tasa de aprendizaje
    loss='binary_crossentropy',           # Perdida para clasificacion binaria
    metrics=['accuracy', Precision(name='precision'),
             Recall(name='recall')]
)

print("\nModel Architecture")
print(model_lstm.summary())

early_stop =  EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose = 1
)

#Training

history_lstm = model_lstm.fit(
    X_train_seq, y_train_seq,
    epochs=50,                # Max 50 Times
    batch_size=32,            # Tamanio de lote
    validation_split=0.2,     # 20% de train
    class_weight=class_weight_dict,    #  Pesos para desbalance
    callbacks=[early_stop],   # callbacks
    verbose=1                 # Mostrar progreso
)
print(f"Epoch Run: {len(history_lstm.history['loss'])}")


Building LSTM Baseline
Model Architecture


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,777 (77.25 KB)

 Trainable params: 19,777 (77.25 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.8015 - loss: 0.4119 - precision: 0.9619 - recall: 0.8234 - val_accuracy: 0.5052 - val_loss: 0.7280 - val_precision: 0.1636 - val_recall: 0.8182
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9381 - loss: 0.2239 - precision: 0.9914 - recall: 0.9429 - val_accuracy: 0.5155 - val_loss: 0.7916 - val_precision: 0.1667 - val_recall: 0.8182
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9510 - loss: 0.1215 - precision: 0.9943 - recall: 0.9538 - val_accuracy: 0.5155 - val_loss: 0.9511 - val_precision: 0.1667 - val_recall: 0.8182
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9716 - loss: 0.0859 - precision: 0.9972 - recall: 0.9728 - val_accuracy: 0.5258 - val_loss: 1.1167 - val_precision: 0.1698 - val_recall: 0.8182
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9794 - loss: 0.0683 - precision: 0.9972 - recall: 0.9810 - val_accuracy: 0.5361 - val_loss: 1.

In [26]:
# Evaluation LSTM

print("Evaluation LSTM")

# Predictions
y_pred_proba_lstm = model_lstm.predict(X_test_seq, verbose=0)

# Convert to binary labels with threshold 0.5
y_pred_lstm = (y_pred_proba_lstm > 0.5).astype(int).flatten()

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

acc_lstm = accuracy_score(y_test_seq, y_pred_lstm)
prec_lstm = precision_score(y_test_seq, y_pred_lstm,zero_division=0)
rec_lstm = recall_score(y_test_seq, y_pred_lstm,zero_division=0)
f1_lstm = f1_score(y_test_seq, y_pred_lstm,zero_division=0)

print("\n LSTM METRICS")
print(f"Accuracy: {acc_lstm:.4f}")
print(f"Precision: {prec_lstm:.4f}")
print(f"Recall: {rec_lstm:.4f}")
print(f"F1 Score: {f1_lstm:.4f}")

print("\n Confusion Matrix:")
cm_lstm = confusion_matrix(y_test_seq, y_pred_lstm)
print(cm_lstm)

print("\n Classification Report")
print(classification_report(y_test_seq, y_pred_lstm, target_names=
 ['Benign', 'Attack']))

#Save Matrics
lstm_metrics = {
    'Model': 'LSTM',
    'Accuracy': acc_lstm,
    'Precision': prec_lstm,
    'Recall': rec_lstm,
    'F1 Score': f1_lstm
}




Evaluation LSTM

 LSTM METRICS
Accuracy: 0.9814
Precision: 0.9912
Recall: 0.9890
F1 Score: 0.9901

 Confusion Matrix:
[[ 28   4]
 [  5 448]]

 Classification Report
              precision    recall  f1-score   support

      Benign       0.85      0.88      0.86        32
      Attack       0.99      0.99      0.99       453

    accuracy                           0.98       485
   macro avg       0.92      0.93      0.93       485
weighted avg       0.98      0.98      0.98       485

